# Seminar HCI and BCI in practice
## Session 2 Simple preprocessing and artifact rejection
---

In [ ]:
# Environment Setting
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import sys

sys.path.append(os.path.join(os.getcwd(), "src"))
from plot_std import plot_std
from remove_bad_epochs import remove_bad_epochs
from multitaper_spectrum import multitaper_spectrum

main_path = os.getcwd()
data_path = os.path.join(main_path, 'data/raw')
print(f'Now you are located: {main_path}')
print(f'Reading and Loading data are located: {data_path}')

---

## Load Baseline Corrected Data

The first preprocessing step is baseline correction, which was done in Session 1

Load data file to workspace (results from session 1 - already baseline corrected)

In [ ]:
# Load prepocessed data from Session 1, and then check the keys from ecog dict
ecog_file = os.path.join(data_path, 'ecogStruct1.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)

# Check data
print(ecog.keys())

---

## Rejecting bad channels

In the next step bad channels are rejected by visualizing their frequency content. The fourier transform of each channel is computed and compared with the mean and the standard deviation of all channels. We need set some parameters at first in the next code cells, after that we will excute a programm `removeBadChannels_periodogram_V2.py` to find out the bad channels.

In [ ]:
# Set sepctrum analysis parameters
params = {
    'tapers': [3, 5],  # TW(time-bandwidth product)=3, K(number of tapers)=5
    'pad': 0,  # no padding
    'Fs': 1000 / ecog['sampDur'],
    'fpass': [0, 200],
    'err': 0,
    'trialavg': False
}

# Multitaper strectral analysis
ecog['data'] = np.array(ecog['data']) # As data originally saved in a list
f, S = multitaper_spectrum(ecog, params)

# Create a new Dict to store multitaper spectral analysis results
periodogram = {
    'trailList': 1,
    'params': params,
    'periodogram':S,
    'centerFrequency':f
}

# update ecog dict
ecog['periodogram'] = periodogram

# Save the spectrual analysised data
with open(os.path.join(data_path, 'ecogStruct1_processed.pkl'), "wb") as file:
    pickle.dump(ecog, file)

# Check again your data
print(ecog.keys())
print(ecog['periodogram'].keys())
for key, value in ecog['periodogram'].items():
    print(f"Key: {key}, Type: {type(value).__name__}")

In [ ]:
ecog_file = os.path.join(data_path, 'ecogStruct1_periodogram.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)

print(ecog.keys())
print(ecog['periodogram'].keys())

<h2 style="color: #FF0000; font-weight: bold;">TASK 1:</h2>

Run the script `src/removeBadChannels_periodogram_V2.py` in **Anaconda Prompt** or **Bash** or **PowerShell** as you did in ```Session_01```. (Alternatively, you can use the `%run` magic command in another code cell: `%run src/removeBadChannels_periodogram_V2.py`. ***But this is not a general solution for all kinds of scripts, for example some functions with users active inputs/operations in terminal.***)

When running the function removeBadChannels a figure will appear (this may take a while!!), showing the mean plus/minus one standard deviation in black lines and a red line showing the frequency spectrum of the first channel. If the channel shows a spectrum considerably exceeding the standard deviation, it should be rejected.

If you succeed with running `removeBadChannels_periodogram_V2.py`, you will see the first channel plot, press ```Good``` to mark the channel as a good channel, press ```Bad``` for bad cahnnels, which later should be rejected. After you click the button, the second channel plot will show up. Do this for all channels. The data will then saved as `ecogStruct1_periodogram.pkl`.

In the end the function will store the indeces of the channels you rejected in the `dict` in  ```ecog['badChannels']```. 

In [ ]:
%run -i src/removeBadChannels_periodogram_V2.py

In [ ]:
# Reload the data after bad channels were marked
ecog_file = os.path.join(data_path, 'ecogStruct1_periodogram.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)
print(f'Channels {ecog['badChannels']} are marked as bad channels')

## Rejection of bad channels by variance estimation

An additional way of identifying bad channels is to look at the standard deviation and then deciding on a minimum and maximum standard deviation.

<h3 style="color: #FF0000; font-weight: bold;">TASK 1.1 (2 Point):</h3>

Calculate the standard deviation over the time series of each channel. 

<h3 style="color: #FF0000; font-weight: bold;">Fill in the missing parts (...) in the code below</h3>

In [ ]:
# standard deviation
ecogdata = np.array(ecog['data'])
# standard deviation
ecogdata = np.array(ecog['data'])
s = ecogdata.std(axis=1)


In [ ]:
plot_std(ecog, s)


In [ ]:
# --- TASK 1.1: look at the numbers behind the plot ---
bad = np.array(ecog['badChannels'])
good = np.setdiff1d(np.arange(1, len(s) + 1), bad)

print("channels marked bad in Task 1 :", bad)
print("std of the good channels      : %.2f - %.2f muV" % (s[good - 1].min(), s[good - 1].max()))
print("std of the bad channels       :", np.round(s[bad - 1], 2))
print("median std over all channels  : %.2f muV" % np.median(s))

print("\nstd of every channel, sorted:")
for ch in np.argsort(s) + 1:
    print(f"  channel {ch:2}: {s[ch - 1]:6.2f}" + ("   <-- marked bad" if ch in bad else ""))

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>
<div style="color: #FF0000; font-weight: bold;">What is a resonable std range(in muV) for a healthy channel? Write the range down, for next task. Why are these marked bad channels rejected?</div>

**A reasonable std range for a healthy channel is about 35 to 55 muV.** I will use `[35, 55]` in the next task.

I got this from the plot and from the numbers. The good channels all lie between **37.96 and 52.51 muV** and the median over all channels is 46.93 muV, so the healthy channels are all quite close together. The two channels that stick out are **channel 31 with 60.42 muV and channel 35 with 60.85 muV**.

The nice thing is that there is an empty gap between the two groups: after 52.51 the next value is already 60.42. So the threshold can be put anywhere in that gap and the result stays the same. I chose 35 and 55 so that there is some space on both sides and the decision does not depend on one single channel.

**Why are the marked bad channels rejected?** In Task 1 I marked **channels 17, 24, 31 and 35**, and they are not all bad for the same reason:

- **Channel 31 and 35** have a clearly too large standard deviation. A large std means the channel is much noisier than the rest, which normally comes from a bad electrode contact or from picking up something that is not brain activity. These two are found by *both* methods, the spectrum and the variance.
- **Channel 17 (49.12 muV) and channel 24 (50.68 muV)** have a completely normal standard deviation. I marked them because their **power spectrum** did not follow the mean plus/minus one std of the other channels. So the variance method would never have found them.

That is exactly why both methods are used: the periodogram finds channels with a wrong *shape* of the spectrum, and the std finds channels that are simply too noisy or too quiet overall.

Rejecting them matters because of what comes next. The data is re-referenced with a common average, so the mean over all channels is subtracted from every channel. If a noisy channel stayed in, its noise would go into that mean and would then be spread into **every other channel**, and the good channels would be made worse too. A channel that is too quiet is a problem as well, because it usually means the electrode had bad contact and is not really recording the brain.

---

<h2 style="color: #FF0000; font-weight: bold;">Task 2:</h2>

Choose the minimum and maximum standard deviation based on the plot.

<h3 style="color: #FF0000; font-weight: bold;">Define min and max threshold for std in the missing parts (...) in the code below</h3>

In [ ]:
# Choose the minimum and maximum standard deviation based on the plot.
minMaxStandardDeviation = [35, 55]     # Set the min and max threshold for std

# Set bad channels and ensure there are no duplicates
idx = np.where((s < minMaxStandardDeviation[0]) | (s > minMaxStandardDeviation[1]))[0]
# idx counts from 0, but badChannels counts from 1, so I have to add 1 before joining them
ecog['badChannels'] = np.unique(np.concatenate([ecog['badChannels'], idx + 1]))
print("channels outside the std range:", idx + 1)
print("all bad channels now          :", ecog['badChannels'])

# Task 2.1 (1 pt):
# Save the numbers of the remaining 'good' channels in the ecog['selectedChannels']
# (hint: check out the documentation for numpy function setdiff1d: https://numpy.org/doc/2.0/reference/generated/numpy.setdiff1d.html)
# --------------------------------------------------------------------------
allChannels = np.arange(1, ecogdata.shape[0] + 1)          # channels 1 ... 40
ecog['selectedChannels'] = np.setdiff1d(allChannels, ecog['badChannels'])
# ==========================================================================
print("number of good channels       :", len(ecog['selectedChannels']))
ecog['selectedChannels']

**Task 2.** I used the range I wrote down in Task 1.1, so `minMaxStandardDeviation = [35, 55]`. Only channel 31 (60.42 muV) and channel 35 (60.85 muV) are outside it, and both of them were already marked bad in Task 1. So the variance method did not find anything new here, it only confirms two of the four channels.

**One thing I had to change in the given code.** `np.where(...)` returns positions that count from **0**, but `ecog['badChannels']` stores channel numbers that count from **1** (the script saves `current_channel + 1`, and `plot_std` also does `badChannels - 1` to use them as positions). If I just join the two lists as it was written, the positions are read as channel numbers and I get

`[17 24 30 31 34 35]`, so 6 bad channels,

and channels 30 and 34 would be thrown away even though they are perfectly fine (their std is 52.51 and 41.03, both inside the range). By writing `idx + 1` both lists count from 1 and the result is

`[17 24 31 35]`, so 4 bad channels.

**Task 2.1.** `np.setdiff1d(allChannels, ecog['badChannels'])` takes all channel numbers from 1 to 40 and removes the ones that are in `badChannels`. What is left are the **36 good channels**, and `setdiff1d` also returns them sorted and without duplicates. This is the list the next steps use, for example the re-referencing, which indexes the data with `ecog['selectedChannels'] - 1`, again because the data itself counts from 0.

---

<h2 style="color: #FF0000; font-weight: bold;">Task 3 (optional):</h2>

***Hint:*** *if you check the online documentation of numpy function `setdiff1d`, you will find the other relevant funtions to solve Task3*

<h3 style="color: #FF0000; font-weight: bold;">Fill in the missing parts (...) in the code below</h3>

In [ ]:
# Define arrays
a = np.array([1, 2, 3, 4, 5])
b = np.array([3, 4, 5, 6, 7])

# Task 3.1: What will c contain if c = setdiff(a, b)?
c = ...
print(f'3.1: Array c now is: {c}\n')

# Task 3.2: If you want c to contain [6, 7] instead, what should you change?
c = ...
print(f'3.2: Target array c is {[6,7]}, Array c now is: {c}\n')

# Task 3.3: What function can you use to get [3,4,5]?
c = ...
print(f'3.3: Target array c is {[3,4,5]}, Array c now is: {c}\n')

# Task 3.4: What function can you use to get [1,2,6,7]?
c = ...
print(f'3.4: Target array c is {[1,2,6,7]}, Array c now is: {c}\n')

# Task 3.5: What function can you use to get [1,2,3,4,5,6,7]?
c = ...
print(f'3.5: Target array c is {[1,2,3,4,5,6,7]}, Array c now is: {c}')


---
## Common Average Reference (CAR)

The last step is removing the common average reference (CAR). Here, the mean across all the selected (good) channels is removed from each channel(at each sampling point).

In [ ]:
# Selecting only the data from the good channels
data = ecog['data'][ecog['selectedChannels'] - 1, :]  # Adjust for 0-based indexing in Python

# Figure: Plot example data before re-referencing
fig, axes = plt.subplots(2, 1, figsize=(10, 5))

# First subplot (Before re-referencing)
axes[0].plot(data[14, 0:10000])  # Just a random channel and time points for demonstration
axes[0].set_title('Before Re-referencing')
axes[0].set_ylabel('Amplitude')

# DO NOT Close the plot window before the next subplot shows up!

<h2 style="color: #FF0000; font-weight: bold;">TASK 4.1 (1 pt):</h2>

<h3 style="color: #FF0000; font-weight: bold;">Fill in the missing parts (...) in the code below</h3>

In [ ]:
# Finding the new reference channel of the time series (TS)
ecog['refChanTS'] = data.mean(axis=0)   # Mean across all channels for each time point
data = data - ecog['refChanTS']         # Subtract this reference Channel from each channel

# small check of what happened
print("refChanTS shape :", ecog['refChanTS'].shape, "-> one value per time point")
print("std of the reference itself : %.2f" % ecog['refChanTS'].std())
print("channel std after CAR       : %.2f - %.2f" % (data.std(axis=1).min(), data.std(axis=1).max()))
print("mean over channels at each time point is now:", np.abs(data.mean(axis=0)).max())

In [ ]:
# Figure: Plot example data after re-referencing
# Second subplot (After re-referencing)
axes[1].plot(data[14, 0:10000])  # Same channel as before, now with CAR removed
axes[1].set_title('After Re-referencing')
axes[1].set_xlabel('Time Points')
axes[1].set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

**Task 4.1.** `data.mean(axis=0)` takes the mean **over the channels**, so I get one value for every time point and the shape is `(522868,)`. That is the common average reference. Then I subtract it from `data`, and because the shape of `data` is `(36, 522868)` numpy broadcasts the reference over all 36 channels automatically, so every channel loses the same value at the same time point.

It is worth comparing this with the baseline correction from Session 1. There I needed `axis=1` and `keepdims=True`, here I need `axis=0` and no `keepdims`:

- **Baseline correction:** mean over the *samples*, one value per channel, so a constant offset is removed from each channel. `keepdims` was needed because the shape `(40,)` does not fit the 522868 samples.
- **CAR:** mean over the *channels*, one value per time point, so the part of the signal that all electrodes see at the same moment is removed. Here the shape `(522868,)` already fits the second axis of `data`, so broadcasting works without `keepdims`.

Note that the mean is taken only over the 36 **good** channels, because `data` was built from `ecog['selectedChannels']` in the cell above. If the bad channels were still in, their noise would go into the reference and would then be subtracted from every good channel.

**What the numbers show.** The reference itself has a std of 40.50, which is as large as the std of a single channel (the good ones are between 37.96 and 52.51). So a big part of what the electrodes record is the same everywhere and is not specific to one place on the cortex. After subtracting it, the std of the channels drops to **15.70 - 35.51**, so roughly half of the variability was shared. And the mean over the channels at every time point is now 0 (1.8e-13, which is just floating point rounding), which is exactly what CAR is supposed to do.

In the figure the trace after re-referencing looks smaller and less smooth than before. The slow movements that all channels had in common are gone, and what is left is more specific to that single electrode.

In [ ]:
# Saving the re-referenced data
ecog['data'][ecog['selectedChannels'] - 1, :] = data  # Save back to ECoG structure

# Save data for the next step
with open(os.path.join(data_path, 'ecogStruct1_periodogram.pkl'), 'wb') as f:
    pickle.dump(ecog, f)

<h2 style="color: #FF0000; font-weight: bold;">TASK 4.2 (Discussion, 1pt):</h2>

Why do we remove the common average reference? What are the advantages of the CAR? Are there disadvantages? Are there alternatives?
s have no brain activity.


<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration:</h3>

In [ ]:
# --- TASK 4.2: how much of the signal was shared between the channels? ---
after  = ecog['data'][ecog['selectedChannels'] - 1, :]   # data as it is now (CAR removed)
before = after + ecog['refChanTS']                       # add the reference back to get the old data

def pair_corr(x):
    c = np.corrcoef(x[:, ::20])                # every 20th sample is enough here
    iu = np.triu_indices_from(c, 1)            # only the upper triangle, no self-correlations
    return c[iu]

cb = pair_corr(before)
ca = pair_corr(after)

print("correlation between channel pairs BEFORE CAR: mean %+.2f  (min %+.2f  max %+.2f)"
      % (cb.mean(), cb.min(), cb.max()))
print("correlation between channel pairs AFTER  CAR: mean %+.2f  (min %+.2f  max %+.2f)"
      % (ca.mean(), ca.min(), ca.max()))
print("pairs that are negative after CAR: %d out of %d" % ((ca < 0).sum(), ca.size))

n = len(ecog['selectedChannels'])
print("\n-1/(N-1) with N = %d channels is %+.3f, and the measured mean is %+.3f"
      % (n, -1 / (n - 1), ca.mean()))

vb = before.var(axis=1).sum()
va = after.var(axis=1).sum()
print("total variance before %.3g -> after %.3g, so %.0f%% was shared and got removed"
      % (vb, va, 100 * (1 - va / vb)))

**Why do we remove the common average reference?**

Every electrode measures a voltage *against a reference*, so whatever happens at the reference is added to all channels. On top of that the electrodes pick up things that are not local brain activity at all: mains noise, the amplifier, movement, and activity that is so widespread that it reaches the whole grid. All of this is the same in every channel at the same moment, so it can be estimated by simply averaging over the channels and then subtracted.

The numbers in the cell above show how big this shared part is here. Before the CAR, **every pair of channels correlates with a mean of +0.76**, and even the least similar pair is still at +0.57. Two electrodes 0.4 cm apart should not look almost identical, so most of that is not local activity. After removing the CAR, **76% of the total variance is gone**, and the reference itself has a std of 40.50, which is as large as a whole channel. So most of what was recorded was common to everything.

**Advantages.**

- It removes noise that all electrodes share, so what is left is much more specific to the place under each electrode.
- It does not need an extra reference electrode, and it does not depend on one single electrode being quiet. A physical reference electrode can itself be noisy, and then that noise goes into every channel.
- It treats all channels equally, so no electrode is favoured, and it is very easy to compute.

**Disadvantages.**

- Every channel goes into the mean, so a bit of each channel ends up in all the others, with the opposite sign. This creates a **negative correlation between the channels that is not real**. In theory it is `-1/(N-1)`, and with our N = 36 channels that is -0.029. The measured mean correlation after CAR is -0.03, so exactly what the formula predicts, and 414 of the 630 channel pairs became negative.
- If the real activity is widespread and not focal, it is also in the mean and gets removed together with the noise. So CAR works well for local activity and can hurt when something happens everywhere at once.
- The result depends on which channels are averaged. That is why the bad channels had to be removed first, in Task 1 to 2.1: a noisy channel in the mean would be spread into all the good ones. It also depends on how the grid is placed, because the mean is only over the part of the cortex that is covered.
- The reference is no longer a fixed physical point, so the values are harder to compare with a recording that used a different montage.

**Alternatives.**

- **Bipolar reference**, where each channel is the difference to its neighbour. This is very local, but the signal then belongs to a pair of electrodes and not to one position.
- **Laplacian reference**, where the mean of the surrounding electrodes is subtracted instead of the mean of all of them. This keeps the local activity better than CAR and is a good option for a dense grid like the 16x16 one here.
- **Median instead of mean**, which is much less sensitive to a single noisy channel, so a bad channel would not damage the reference as much.
- **A dedicated reference electrode** somewhere without brain activity, for example on the bone. This is a real reference, but only works if that electrode is really quiet.
- **Removing the shared part by regression or ICA**, which finds the common components in the data instead of assuming that the simple average is the right one.

---

## Visual artefact removal

Finally remove artifacts by visual inspection of the remaining time series. 

<h2 style="color: #FF0000; font-weight: bold;">TASK 5(1 Pt):</h2>

```ecog_gui_v3.py```

To visually inspect the remaining time series the ```ecog_gui_v3.py (which is a TSGUI(TimeSeries Graphical User Interface))``` can be used. Change the number of viewed channels to make things a bit clearer. Then go through all the time series. If you find a bad (noisy) interval, press and hold `SHIFT` key to select [mouse click] the start and the end of the interval in the time series. **When it turns blue, press 'b' to add the interval to the list of bad intervals.** 


<h4 style="color: #0000FF; font-weight: bold;">GUI Manual:</h4>

**Ch**(value input): user can define which channels should be ploted in the interface, default [1, 40]

**Start**(value input): user define the left edge of the plot, default 0

**\#**(value input): user define how many channels will show up in the interface, default 40

**Interval**(value input): user define how many seconds are currently ploted in the interface, default 5

**up/down button**(functional button): used with **\#**, if there are only 20 channels in the interface now, use buttons to scroll up/down

**<</>> button**(functional button): scroll right/left with whole **interval**, if **interval=5**, then go right/left 5 seconds

**</> button**(functional button): scroll right/left with $\frac{1}{3}$ **interval**, if **interval=5**, then go right/left $\frac{5}{3}$ seconds

**vertical scale**(value input/functional button): lower the text `Vertical scale` area needs a user input, to define vertical scaling factor, defaut 1. **\*2 \\2 Button** used to halve or duplicate the scaling factor

**Interaction behavior**: user press **`SHIFT` on keyboard**, then user is able to use **mouse left click** to mark a point, which is selected to be the start time. **Holding on the `SHIFT` button**, user can click the second timepoint. Then this area will be **highlighted in blue**, by **pressing `B` on keyboard**, this interval will be saved and user can continue to mark next interval. **`ESC` on keyboard**, clear unsaved points(before **'B'** pressed). **Click `X(close button)` on the interface** to exit. Data will then saved into `ecogStruct1_badEpochs.pkl`.


In [ ]:
%run ecog_gui_v3.py

In [ ]:
# Load the data with marked bad intervals
ecog_file = os.path.join(data_path, 'ecogStruct1_badEpochs.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)

# Check data 
print(ecog.keys())

In [ ]:
# Sort bad intervals in ascending order
# making sure the bad intervals are in ascending order (in case they were marked in a different order)
selectedIntervalsInGUIUnits = np.array(ecog['marked_intervals']) # Ensure it's a NumPy array
sorted_indices = np.argsort(selectedIntervalsInGUIUnits[:, 0])  # Sort by the first column
selectedIntervalsInGUIUnits = selectedIntervalsInGUIUnits[sorted_indices]

# Add bad intervals to ecog structure
ecog['badIntervals'] = selectedIntervalsInGUIUnits

ecog['badIntervals']

In [ ]:
## Removing 'bad' epochs from the epoch structure

# Load file with epoch information
# provides onsets and labels of gesture epochs (hand labeled) (see Session 1)
epoch_file = os.path.join(data_path, 'epoch.pkl')
with open(epoch_file, 'rb') as f:
    epoch = pickle.load(f)

print(epoch.keys())
print(f'Before moving bad epochs, number of epochs: {len(epoch['OnsetIdx'])}')

# remove epochs overlapping with bad intervals
epoch = remove_bad_epochs(epoch,selectedIntervalsInGUIUnits)
print(f'After moving bad epochs, number of epochs: {epoch['OnsetIdx'].shape}')

In [ ]:
# Save current results 
epoch_file = os.path.join(data_path, 'epoch2.pkl')
with open(epoch_file, 'wb') as f:
    pickle.dump(epoch, f)
ecog_file = os.path.join(data_path, 'ecogStruct2.pkl')
with open(ecog_file, 'wb') as f:
    pickle.dump(ecog, f)

---
<h2 style="color: #FF0000; font-weight: bold;">Additional Task:</h2>

If you have some time left, have a closer look at the `remove_bad_epochs` function. How is the data from the GUI transformed in order to compare it to the onset information stored in the epoch structure? 

---